In [1]:
import pandas as pd
import numpy as np
import polars as pl
from sklearn.metrics import mutual_info_score
from sklearn.model_selection import train_test_split
from IPython.display import display
from datetime import datetime
import os

In [2]:
mes_proceso = None  

In [3]:
df = pl.read_parquet("data/deuda_old.parquet").to_pandas()
actividades=pd.read_excel("data/actividades.xlsx")
actividades['actividad']=actividades['actividad'].astype(str)
del actividades['descripcion']


df=df.merge(actividades,how='left',on='actividad')

In [4]:
df.columns = df.columns.str.lower().str.replace(" ", "_")
categorical_columns = list(df.dtypes[df.dtypes == "str"].index)
#for c in categorical_columns:
#    df[c] = df[c].str.lower().str.replace(" ", "_")

#asegurar columna de periodo en formato datetime
if 'periodo' in df.columns:
    df['periodo'] = pd.to_datetime(df['periodo'])
elif 'fecha' in df.columns:
    df['periodo'] = pd.to_datetime(df['fecha'])

#filtrar por mes_proceso si fue especificado
if mes_proceso is not None:
    df = df[df['periodo'] == pd.to_datetime(mes_proceso)].copy()

df.head()

,periodo,nombre_entidad,situacion,provincia,sexo,tipo_persona,rango_etario,prestamos,mediana_prestamos,cantidad_deudores,rango_prestamo,situacion_mora,descripcion,entidad,actividad,rubro
0,2026-05-01,Naldo Lombardi S.A.,5,Mendoza,M,Física,55-64,20379.0,243.0,77,100-500,En mora,Sin actividad,55041,NaN,NaN
1,2026-05-01,REGGIO DI CALABRIA S.R.L.,1,Tucumán,M,Física,35-44,2290.0,348.0,7,100-500,Normal/Bajo riesgo,Sin actividad,55314,NaN,NaN
2,2026-05-01,Naldo Lombardi S.A.,1,La Pampa,F,Física,<25,140883.0,1738.0,73,1000-5000,Normal/Bajo riesgo,Sin actividad,55041,NaN,NaN
3,2026-05-01,REGGIO DI CALABRIA S.R.L.,1,San Juan,F,Física,35-44,231.0,35.0,7,10-50,Normal/Bajo riesgo,Sin actividad,55314,NaN,NaN
4,2026-05-01,BANCO DE LA PROVINCIA DE CORDOBA S.A.,1,Córdoba,F,Física,>65,24656126.0,6502.0,3623,5000-10000,Normal/Bajo riesgo,Sin actividad,00020,NaN,NaN


In [5]:
df.isnull().sum()

periodo                    0
nombre_entidad           724
situacion                  0
provincia                  0
sexo                       0
tipo_persona               0
rango_etario               0
prestamos                  0
mediana_prestamos          0
cantidad_deudores          0
rango_prestamo             0
situacion_mora             0
descripcion                0
entidad                  236
actividad             803531
rubro                1367391
dtype: int64

In [6]:
df['situacion_mora'] = (df['situacion_mora'] == 'En mora').astype(int)
df['personas_en_mora'] = df['situacion_mora'] * df['cantidad_deudores']
df.tail()

,periodo,nombre_entidad,situacion,provincia,sexo,tipo_persona,rango_etario,prestamos,mediana_prestamos,cantidad_deudores,rango_prestamo,situacion_mora,descripcion,entidad,actividad,rubro,personas_en_mora
1571897,2026-07-01,BANCO COMAFI SOCIEDAD ANONIMA,1,Sin Identificar,Empresa,Jurídica,N/A (Jurídica),43.0,43.0,1,10-50,0,Sin actividad,00299,NaN,NaN,0
1571898,2026-07-01,COMERPYSRL,1,Sin Identificar,Empresa,Física,Sin dato,73.0,73.0,1,50-100,0,Sin actividad,55376,N/A,NaN,0
1571899,2026-07-01,sistemas unificados de credito dirigido SA,5,Sin Identificar,Empresa,Física,Sin dato,67.0,67.0,1,50-100,1,Sin actividad,55547,N/A,NaN,1
1571900,2026-07-01,BRUBANK S.A.U.,1,Sin Identificar,Empresa,Jurídica,N/A (Jurídica),51.0,51.0,1,50-100,0,Sin actividad,00143,NaN,NaN,0
1571901,2026-07-01,JOHN DEERE CREDIT COMPAÑÍA FINANCIERA S.A.,4,Sin Identificar,Empresa,Jurídica,N/A (Jurídica),619203.0,619203.0,1,>10000,1,Sin actividad,44096,NaN,NaN,1


In [7]:
totales_por_periodo = df.groupby('periodo')[['personas_en_mora', 'cantidad_deudores']].sum()
tasa_global_mora = totales_por_periodo['personas_en_mora'] / totales_por_periodo['cantidad_deudores']
display(tasa_global_mora.rename("tasa_global_mora"))

periodo
2026-05-01    0.350987
2026-06-01    0.246102
2026-07-01    0.241656
Name: tasa_global_mora, dtype: float64

In [8]:
mora_sexo = df.groupby(['periodo', 'sexo']).apply(
    lambda x: (x['situacion_mora'] * x['cantidad_deudores']).sum() / x['cantidad_deudores'].sum(),
    include_groups=False
).unstack()

display(mora_sexo)

sexo,Empresa,F,M,X
periodo,,,,
2026-05-01,0.186789,0.339212,0.369188,0.423077
2026-06-01,0.144822,0.237804,0.257758,0.239609
2026-07-01,0.146073,0.234214,0.252335,0.232893


In [9]:
df_group = df.groupby(['periodo', 'sexo'])[['personas_en_mora', 'cantidad_deudores']].sum().reset_index()

df_group['mean'] = df_group['personas_en_mora'] / df_group['cantidad_deudores']
df_group['count'] = df_group['cantidad_deudores']

df_group['tasa_ref'] = df_group['periodo'].map(tasa_global_mora)
df_group['diff'] = df_group['mean'] - df_group['tasa_ref']
df_group['risk'] = df_group['mean'] / df_group['tasa_ref']

df_group = df_group[['periodo', 'sexo', 'mean', 'count', 'diff', 'risk']]
display(df_group)

,periodo,sexo,mean,count,diff,risk
0,2026-05-01,Empresa,0.186789,38787,-0.164198,0.532183
1,2026-05-01,F,0.339212,1106734,-0.011776,0.966450
2,2026-05-01,M,0.369188,1065889,0.018200,1.051855
3,2026-05-01,X,0.423077,26,0.072090,1.205391
4,2026-06-01,Empresa,0.144822,520583,-0.101279,0.588465
5,2026-06-01,F,0.237804,20949151,-0.008297,0.966284
6,2026-06-01,M,0.257758,19435677,0.011657,1.047365
7,2026-06-01,X,0.239609,818,-0.006493,0.973617
8,2026-07-01,Empresa,0.146073,523922,-0.095583,0.604467
9,2026-07-01,F,0.234214,20616223,-0.007442,0.969203


In [10]:
df.columns

Index(['periodo', 'nombre_entidad', 'situacion', 'provincia', 'sexo',
       'tipo_persona', 'rango_etario', 'prestamos', 'mediana_prestamos',
       'cantidad_deudores', 'rango_prestamo', 'situacion_mora', 'descripcion',
       'entidad', 'actividad', 'rubro', 'personas_en_mora'],
      dtype='str')

In [12]:
df['provincia'].unique()

<ArrowStringArray>
[            'Mendoza',             'Tucumán',            'La Pampa',
            'San Juan',             'Córdoba',        'Buenos Aires',
            'Santa Fe', 'Santiago del Estero',          'Corrientes',
          'Entre Ríos',            'Misiones',               'Salta',
                'CABA',             'Neuquén',               'Chaco',
            'La Rioja',             'Formosa',               'Jujuy',
            'San Luis',           'Río Negro',          'Santa Cruz',
           'Catamarca',              'Chubut',    'Tierra del Fuego',
     'Sin Identificar']
Length: 25, dtype: str

In [13]:
categorical = ['provincia', 'sexo', 'tipo_persona', 'rango_etario', 'descripcion','rubro', 'actividad']
numeric = ['personas_en_mora', 'prestamos', 'cantidad_deudores']
provincias = [
    'Santa Fe', 'Buenos Aires', 'San Juan', 'Catamarca', 'Salta', 'Río Negro',
    'Misiones', 'San Luis', 'Córdoba', 'Tucumán', 'Mendoza', 'Chaco',
    'CABA', 'Entre Ríos', 'La Pampa', 'Neuquén', 'Chubut', 'Santa Cruz',
    'Jujuy', 'Tierra del Fuego', 'Corrientes', 'Santiago del Estero', 'La Rioja', 'Formosa',
    'Sin Identificar'
]

In [14]:
df[categorical].nunique()

provincia         25
sexo               4
tipo_persona       3
rango_etario       8
descripcion     1066
rubro             21
actividad       1071
dtype: int64

In [15]:
resultados_globales = []

for c in categorical:
    print(f"Calculando métricas globales para: {c}")
    
    #periodo y entidad
    df_ent = df.groupby(['periodo', 'entidad', c])[['personas_en_mora', 'cantidad_deudores', 'prestamos']].sum().reset_index()
    df_ent['mean'] = df_ent['personas_en_mora'] / df_ent['cantidad_deudores']
    df_ent['count'] = df_ent['cantidad_deudores']
    df_ent['prestamos'] = df_ent['prestamos']
    df_ent['tasa_ref'] = df_ent['periodo'].map(tasa_global_mora)
    df_ent['diff'] = df_ent['mean'] - df_ent['tasa_ref']
    df_ent['risk'] = df_ent['mean'] / df_ent['tasa_ref']
    df_ent.rename(columns={c: 'caracteristica'}, inplace=True)
    df_ent['columna'] = c
    df_ent['provincia'] = 'global'
    resultados_globales.append(df_ent)

    #general
    df_all = df.groupby(['periodo', c])[['personas_en_mora', 'cantidad_deudores', 'prestamos']].sum().reset_index()
    df_all['mean'] = df_all['personas_en_mora'] / df_all['cantidad_deudores']
    df_all['count'] = df_all['cantidad_deudores']
    df_all['prestamos'] = df_all['prestamos']
    df_all['tasa_ref'] = df_all['periodo'].map(tasa_global_mora)
    df_all['diff'] = df_all['mean'] - df_all['tasa_ref']
    df_all['risk'] = df_all['mean'] / df_all['tasa_ref']
    df_all.rename(columns={c: 'caracteristica'}, inplace=True)
    df_all['columna'] = c
    df_all['provincia'] = 'global'
    df_all['entidad'] = 'todas'
    resultados_globales.append(df_all)

df_metricas_global = pd.concat(resultados_globales, ignore_index=True)
df_metricas_global = df_metricas_global[['periodo', 'entidad', 'provincia', 'columna', 'caracteristica', 'mean', 'count', 'prestamos', 'diff', 'risk']]

Calculando métricas globales para: provincia
Calculando métricas globales para: sexo
Calculando métricas globales para: tipo_persona
Calculando métricas globales para: rango_etario
Calculando métricas globales para: descripcion
Calculando métricas globales para: rubro
Calculando métricas globales para: actividad


In [16]:
resultados_globales_provincia = []

for p in provincias:
    print(f"Procesando provincia: {p}...")
    df_prov = df[df['provincia'] == p].copy()
    if df_prov.empty:
        continue

    for c in categorical:
        #agrupación por periodo, entidad y provincia
        df_ent = df_prov.groupby(['periodo', 'entidad', c])[['personas_en_mora', 'cantidad_deudores', 'prestamos']].sum().reset_index()
        df_ent['mean'] = df_ent['personas_en_mora'] / df_ent['cantidad_deudores']
        df_ent['count'] = df_ent['cantidad_deudores']
        df_ent['prestamos'] = df_ent['prestamos']
        df_ent['tasa_ref'] = df_ent['periodo'].map(tasa_global_mora)
        df_ent['diff'] = df_ent['mean'] - df_ent['tasa_ref']
        df_ent['risk'] = df_ent['mean'] / df_ent['tasa_ref']
        df_ent.rename(columns={c: 'caracteristica'}, inplace=True)
        df_ent['provincia'] = p
        df_ent['columna'] = c
        resultados_globales_provincia.append(df_ent)

        #consolidado provincia (todas las entidades)
        df_all = df_prov.groupby(['periodo', c])[['personas_en_mora', 'cantidad_deudores', 'prestamos']].sum().reset_index()
        df_all['mean'] = df_all['personas_en_mora'] / df_all['cantidad_deudores']
        df_all['count'] = df_all['cantidad_deudores']
        df_all['prestamos'] = df_all['prestamos']
        df_all['tasa_ref'] = df_all['periodo'].map(tasa_global_mora)
        df_all['diff'] = df_all['mean'] - df_all['tasa_ref']
        df_all['risk'] = df_all['mean'] / df_all['tasa_ref']
        df_all.rename(columns={c: 'caracteristica'}, inplace=True)
        df_all['provincia'] = p
        df_all['columna'] = c
        df_all['entidad'] = 'todas'
        resultados_globales_provincia.append(df_all)

df_metricas_provincia = pd.concat(resultados_globales_provincia, ignore_index=True)
df_metricas_provincia = df_metricas_provincia[['periodo', 'entidad', 'provincia', 'columna', 'caracteristica', 'mean', 'count', 'prestamos', 'diff', 'risk']]

df_metricas_total = pd.concat([df_metricas_global, df_metricas_provincia], ignore_index=True)
df_metricas_total.rename(columns={'periodo': 'fecha'}, inplace=True)
df_metricas_total['fecha'] = pd.to_datetime(df_metricas_total['fecha'])

ruta_archivo = 'data/metricas_totales.parquet'

if os.path.exists(ruta_archivo):
    print("Archivo histórico encontrado. Leyendo datos anteriores...")
    df_historico = pl.read_parquet(ruta_archivo).to_pandas()
    df_historico['fecha'] = pd.to_datetime(df_historico['fecha'])
    
    # Compatibilidad: si el histórico no tenía la columna entidad, asignar 'todas'
    if 'entidad' not in df_historico.columns:
        df_historico['entidad'] = 'todas'
    
    # Control conjunto por periodo (fecha) y entidad para evitar duplicados
    claves_nuevas = df_metricas_total[['fecha', 'entidad']].drop_duplicates()
    df_historico_filtrado = df_historico.merge(claves_nuevas, on=['fecha', 'entidad'], how='left', indicator=True)
    df_historico_filtrado = df_historico_filtrado[df_historico_filtrado['_merge'] == 'left_only'].drop(columns=['_merge'])
    
    df_final = pd.concat([df_historico_filtrado, df_metricas_total], ignore_index=True)
else:
    print("No se encontró archivo histórico. Se creará uno nuevo.")
    df_final = df_metricas_total

df_final.to_parquet(ruta_archivo, index=False)
print(f"Proceso finalizado. El dataset histórico ahora tiene {len(df_final)} filas.")

Procesando provincia: Santa Fe...
Procesando provincia: Buenos Aires...
Procesando provincia: San Juan...
Procesando provincia: Catamarca...
Procesando provincia: Salta...
Procesando provincia: Río Negro...
Procesando provincia: Misiones...
Procesando provincia: San Luis...
Procesando provincia: Córdoba...
Procesando provincia: Tucumán...
Procesando provincia: Mendoza...
Procesando provincia: Chaco...
Procesando provincia: CABA...
Procesando provincia: Entre Ríos...
Procesando provincia: La Pampa...
Procesando provincia: Neuquén...
Procesando provincia: Chubut...
Procesando provincia: Santa Cruz...
Procesando provincia: Jujuy...
Procesando provincia: Tierra del Fuego...
Procesando provincia: Corrientes...
Procesando provincia: Santiago del Estero...
Procesando provincia: La Rioja...
Procesando provincia: Formosa...
Procesando provincia: Sin Identificar...
No se encontró archivo histórico. Se creará uno nuevo.
Proceso finalizado. El dataset histórico ahora tiene 633759 filas.


In [17]:
categorical_cruzado = ['descripcion', 'sexo', 'rango_etario', 'tipo_persona']
columnas_prov_ent = ['periodo', 'entidad', 'provincia'] + categorical_cruzado

# 1. Por provincia y entidad
df_cruzado_ent = df.groupby(columnas_prov_ent)[['personas_en_mora', 'cantidad_deudores', 'prestamos']].sum().reset_index()
df_cruzado_ent['mean'] = df_cruzado_ent['personas_en_mora'] / df_cruzado_ent['cantidad_deudores']
df_cruzado_ent['count'] = df_cruzado_ent['cantidad_deudores']
df_cruzado_ent['tasa_ref'] = df_cruzado_ent['periodo'].map(tasa_global_mora)
df_cruzado_ent['diff'] = df_cruzado_ent['mean'] - df_cruzado_ent['tasa_ref']
df_cruzado_ent['risk'] = df_cruzado_ent['mean'] / df_cruzado_ent['tasa_ref']
df_cruzado_ent = df_cruzado_ent[columnas_prov_ent + ['mean', 'count', 'prestamos', 'diff', 'risk']]

# 2. Por provincia consolidado (todas las entidades)
columnas_prov_all = ['periodo', 'provincia'] + categorical_cruzado
df_cruzado_all = df.groupby(columnas_prov_all)[['personas_en_mora', 'cantidad_deudores', 'prestamos']].sum().reset_index()
df_cruzado_all['mean'] = df_cruzado_all['personas_en_mora'] / df_cruzado_all['cantidad_deudores']
df_cruzado_all['count'] = df_cruzado_all['cantidad_deudores']
df_cruzado_all['tasa_ref'] = df_cruzado_all['periodo'].map(tasa_global_mora)
df_cruzado_all['diff'] = df_cruzado_all['mean'] - df_cruzado_all['tasa_ref']
df_cruzado_all['risk'] = df_cruzado_all['mean'] / df_cruzado_all['tasa_ref']
df_cruzado_all['entidad'] = 'todas'
df_cruzado_all = df_cruzado_all[columnas_prov_ent + ['mean', 'count', 'prestamos', 'diff', 'risk']]

# 3. Global por entidad
columnas_glob_ent = ['periodo', 'entidad'] + categorical_cruzado
df_glob_ent = df.groupby(columnas_glob_ent)[['personas_en_mora', 'cantidad_deudores', 'prestamos']].sum().reset_index()
df_glob_ent['mean'] = df_glob_ent['personas_en_mora'] / df_glob_ent['cantidad_deudores']
df_glob_ent['count'] = df_glob_ent['cantidad_deudores']
df_glob_ent['tasa_ref'] = df_glob_ent['periodo'].map(tasa_global_mora)
df_glob_ent['diff'] = df_glob_ent['mean'] - df_glob_ent['tasa_ref']
df_glob_ent['risk'] = df_glob_ent['mean'] / df_glob_ent['tasa_ref']
df_glob_ent['provincia'] = 'global'
df_glob_ent = df_glob_ent[columnas_prov_ent + ['mean', 'count', 'prestamos', 'diff', 'risk']]

# 4. Global consolidado (todas las entidades)
columnas_glob_all = ['periodo'] + categorical_cruzado
df_glob_all = df.groupby(columnas_glob_all)[['personas_en_mora', 'cantidad_deudores', 'prestamos']].sum().reset_index()
df_glob_all['mean'] = df_glob_all['personas_en_mora'] / df_glob_all['cantidad_deudores']
df_glob_all['count'] = df_glob_all['cantidad_deudores']
df_glob_all['tasa_ref'] = df_glob_all['periodo'].map(tasa_global_mora)
df_glob_all['diff'] = df_glob_all['mean'] - df_glob_all['tasa_ref']
df_glob_all['risk'] = df_glob_all['mean'] / df_glob_all['tasa_ref']
df_glob_all['provincia'] = 'global'
df_glob_all['entidad'] = 'todas'
df_glob_all = df_glob_all[columnas_prov_ent + ['mean', 'count', 'prestamos', 'diff', 'risk']]

# Concatenar todos los cruces
df_metricas_total_cruzado = pd.concat([df_glob_all, df_glob_ent, df_cruzado_all, df_cruzado_ent], ignore_index=True)
df_metricas_total_cruzado.rename(columns={'periodo': 'fecha'}, inplace=True)
df_metricas_total_cruzado['fecha'] = pd.to_datetime(df_metricas_total_cruzado['fecha'])

ruta_archivo_cruzado = 'data/metricas_totales_cruzadas.parquet'

if os.path.exists(ruta_archivo_cruzado):
    print("Archivo histórico encontrado. Actualizando datos...")
    df_historico_cruzado = pl.read_parquet(ruta_archivo_cruzado).to_pandas()
    df_historico_cruzado['fecha'] = pd.to_datetime(df_historico_cruzado['fecha'])

    if 'entidad' not in df_historico_cruzado.columns:
        df_historico_cruzado['entidad'] = 'todas'

    # Control conjunto por periodo (fecha) y entidad
    claves_nuevas_cruzado = df_metricas_total_cruzado[['fecha', 'entidad']].drop_duplicates()
    df_historico_cruzado_filtrado = df_historico_cruzado.merge(claves_nuevas_cruzado, on=['fecha', 'entidad'], how='left', indicator=True)
    df_historico_cruzado_filtrado = df_historico_cruzado_filtrado[df_historico_cruzado_filtrado['_merge'] == 'left_only'].drop(columns=['_merge'])

    df_final_cruzado = pd.concat([df_historico_cruzado_filtrado, df_metricas_total_cruzado], ignore_index=True)
else:
    print("No se encontró archivo histórico. Se creará uno nuevo.")
    df_final_cruzado = df_metricas_total_cruzado

df_final_cruzado.to_parquet(ruta_archivo_cruzado, index=False)
print(f"Proceso finalizado. El dataset histórico cruzado ahora tiene {len(df_final_cruzado)} filas.")

No se encontró archivo histórico. Se creará uno nuevo.
Proceso finalizado. El dataset histórico cruzado ahora tiene 418770 filas.


In [22]:
variables_predictoras = ['nombre_entidad', 'provincia', 'sexo', 'tipo_persona', 'rango_etario', 'descripcion']
target = 'situacion_mora'
frecuencia = 'cantidad_deudores'

periodos_unicos = sorted(df['periodo'].unique())
registros_mi = []

for p in periodos_unicos:
    df_p = df[df['periodo'] == p]
    for var in variables_predictoras:
        contingency = df_p.pivot_table(
            index=var,
            columns=target,
            values=frecuencia,
            aggfunc='sum',
            fill_value=0
        ).values
        score = mutual_info_score(None, None, contingency=contingency)
        registros_mi.append({
            'periodo': p.strftime('%Y-%m') if hasattr(p, 'strftime') else str(p),
            'variable': var,
            'mutual_information': score
        })

df_importancia_periodo = pd.DataFrame(registros_mi)
tabla_mi = df_importancia_periodo.pivot(index='variable', columns='periodo', values='mutual_information')
ultimo_periodo = tabla_mi.columns[-1]
tabla_mi = tabla_mi.sort_values(by=ultimo_periodo, ascending=False)

display(tabla_mi)

periodo,2026-05,2026-06,2026-07
variable,,,
nombre_entidad,0.243054,0.088888,0.080454
rango_etario,0.043119,0.012929,0.012928
provincia,0.034463,0.004065,0.004133
descripcion,0.002528,0.000723,0.000696
sexo,0.001651,0.000663,0.000587
tipo_persona,0.001797,0.000517,0.000480
